# Part 2 - RAG Knowledge Base for Dynamics 365 F&O

## Goal

The planning document says the Gap and Retrieval Agent *"searches approved knowledge"* and that the
planning, design and testing agents generate their artefacts *"using retrieved organizational memory"*.
It also says that knowledge search is exposed through an **MCP server**.

This notebook builds that knowledge layer:

1. Store the organization's approved knowledge as documents with metadata: **Microsoft guidance**,
   **company standards**, **previously approved designs** and **email templates**.
2. Split the documents into chunks and convert them into embeddings.
3. Store the embeddings in a vector store and search with **metadata filters** (module, source type,
   approved only).
4. Answer questions using only the retrieved context, with **citations**.
5. Expose the search as an **MCP server** and let an agent use it through the protocol.

## Setup

Same environment as the course notebooks: a local **Ollama** server running **`gemma4:e4b`** for the agents and **`nomic-embed-text`** for embeddings. The cells work on Google Colab (Ollama is installed automatically) and locally (install Ollama from ollama.com first).

This notebook uses both the chat model and the embedding model.

**Chat model:** set `LLM_PROVIDER=openai` in `.env` to run the agents on **GPT Luna** (`OPENAI_MODEL`, default `gpt-6-luna`, key in `OPEN_AI_KEY`). The embeddings stay on Ollama (`nomic-embed-text`), so the saved knowledge store keeps working with either chat model. `LLM_PROVIDER=ollama` (the default) keeps the local `gemma4:e4b`.

In [ ]:
%pip install -q -U langchain langchain-core langchain-ollama langchain-openai python-dotenv langchain-text-splitters langgraph langchain-mcp-adapters mcp python-docx requests numpy langsmith

In [ ]:
import os
import platform
import shutil
import subprocess
import time

import requests
from dotenv import load_dotenv

load_dotenv()  # reads .env next to the notebook (OPEN_AI_KEY, LLM_PROVIDER, ADO_*, ...) when it exists

# Chat model: "openai" runs the agents on GPT Luna (key in OPEN_AI_KEY), "ollama" on the local model.
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "ollama").strip().lower()
if LLM_PROVIDER not in ("openai", "ollama"):
    raise ValueError(f"LLM_PROVIDER must be 'openai' or 'ollama', not {LLM_PROVIDER!r}")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "gemma4:e4b")
LLM_MODEL = OPENAI_MODEL if LLM_PROVIDER == "openai" else OLLAMA_MODEL
EMBED_MODEL = os.getenv("EMBED_MODEL", "nomic-embed-text")
OLLAMA_URL = "http://localhost:11434"
NEEDS_OLLAMA = True  # the embeddings always run on Ollama


def ollama_is_running() -> bool:
    try:
        return requests.get(f"{OLLAMA_URL}/api/tags", timeout=3).ok
    except requests.RequestException:
        return False


# 1. Install Ollama (Colab / Linux) and start the server in the background
if NEEDS_OLLAMA:
    if shutil.which("ollama") is None:
        if platform.system() == "Linux":
            print("Installing Ollama...")
            !sudo apt-get install -y -qq zstd
            !curl -fsSL https://ollama.com/install.sh | sh
        else:
            print("Install Ollama from https://ollama.com/download, then re-run this cell.")

    if not ollama_is_running():
        print("Starting Ollama server...")
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(5)  # Wait for server to wake up

    print("Ollama running:", ollama_is_running())
print(f"Chat model: {LLM_PROVIDER} / {LLM_MODEL}")

In [ ]:
# 2. Pull the chat model (only when the agents run on Ollama) and the embedding model
if LLM_PROVIDER == "ollama":
    !ollama pull {LLM_MODEL}
!ollama pull {EMBED_MODEL}
!ollama list

### LangSmith tracing

Every agent, LLM call, MCP tool call and LangGraph node is traced in **LangSmith**, so you can open each step
and see its **prompt, output, duration and tokens**. Paste your API key when asked (or store it as the Colab
secret `LANGSMITH_API_KEY`); press Enter to run without tracing.

In [ ]:
import getpass
import os

import langsmith.utils

# LangSmith records every agent, LLM call (full prompt and answer), MCP tool call and LangGraph node,
# with its duration and token usage. Free API key: https://smith.langchain.com -> Settings -> API Keys.
# Note: traces contain the prompts, so customer emails are sent to LangSmith. Use test data or your own workspace.
USE_LANGSMITH = True
LANGSMITH_PROJECT = "fo-requirements-to-delivery"


def read_langsmith_key() -> str:
    if os.getenv("LANGSMITH_API_KEY"):
        return os.environ["LANGSMITH_API_KEY"]
    try:
        from google.colab import userdata  # Colab: add a secret named LANGSMITH_API_KEY

        return userdata.get("LANGSMITH_API_KEY")
    except Exception:
        pass
    try:
        return getpass.getpass("LangSmith API key (press Enter to run without tracing): ").strip()
    except Exception:  # no interactive input available
        return ""


api_key = read_langsmith_key() if USE_LANGSMITH else ""
if api_key:
    os.environ.update(LANGSMITH_API_KEY=api_key, LANGSMITH_TRACING="true", LANGSMITH_PROJECT=LANGSMITH_PROJECT)
    print(f"LangSmith tracing ON -> project '{LANGSMITH_PROJECT}' on https://smith.langchain.com")
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith tracing OFF")
langsmith.utils.get_env_var.cache_clear()  # LangSmith caches environment variables

In [ ]:
import getpass
import os


def make_llm():
    """The chat model selected by LLM_PROVIDER: GPT Luna on OpenAI, or the local Ollama model."""
    if LLM_PROVIDER == "openai":
        from langchain_openai import ChatOpenAI

        if not (os.getenv("OPEN_AI_KEY") or os.getenv("OPENAI_API_KEY")):
            os.environ["OPEN_AI_KEY"] = getpass.getpass("OpenAI API key: ").strip()
        # GPT Luna only accepts its default temperature, so none is set.
        return ChatOpenAI(model=LLM_MODEL, api_key=os.getenv("OPEN_AI_KEY") or os.getenv("OPENAI_API_KEY"))

    from langchain_ollama import ChatOllama

    return ChatOllama(model=LLM_MODEL, temperature=0)


llm = make_llm()

In [ ]:
import json

from langchain_core.messages import HumanMessage, SystemMessage
from langsmith import traceable  # names agent steps in LangSmith (does nothing when tracing is off)


def ask_structured(schema, system_prompt: str, user_prompt: str, retries: int = 2):
    """Call the LLM and force the answer into a Pydantic schema. On failure, retry with the error message."""
    # The run name shows up in LangSmith, e.g. "DeliveryPlan" or "GapAnalysis".
    structured_llm = llm.with_structured_output(schema, method="json_schema").with_config(run_name=schema.__name__)
    last_error = None

    for attempt in range(retries + 1):
        try:
            result = structured_llm.invoke(
                [SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]
            )
            if result is None:
                raise ValueError("The model returned an empty answer.")
            return result
        except Exception as error:
            last_error = error
            print(f"[retry {attempt + 1}] invalid structured output: {error}")
            user_prompt += (
                f"\n\nYour previous answer was invalid ({error}). "
                "Return JSON that matches the schema exactly."
            )

    raise last_error


def ask_text(system_prompt: str, user_prompt: str) -> str:
    return llm.invoke(
        [SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]
    ).text  # plain string for both providers (OpenAI returns content blocks)


def show(title: str, obj) -> None:
    """Pretty-print a heading and a Pydantic object, dict, list or string."""
    print(f"\n========== {title} ==========\n")
    if hasattr(obj, "model_dump"):
        obj = obj.model_dump()
    print(obj if isinstance(obj, str) else json.dumps(obj, indent=2, ensure_ascii=False))

## The Knowledge Documents

Each document has metadata that the agents rely on:

- `doc_id` - used as a citation, for example `[MS-WF-01]`
- `source_type` - `microsoft_guidance`, `company_standard`, `approved_design` or `email_template`
- `module` - `Finance`, `Supply Chain` or `General`
- `approved` - only approved knowledge may be used. `DES-DRAFT-007` is a rejected draft that is kept in
  the folder on purpose, to prove the filter works.

The Microsoft entries are short summaries written for this project, not copies of the documentation.

In [ ]:
# Summarised notes used as the organization's approved knowledge.
# source_type: microsoft_guidance | company_standard | approved_design | email_template
KNOWLEDGE_DOCS = [
    # ---------------- Microsoft guidance (summarised) ----------------
    {"doc_id": "MS-EXT-01", "title": "Extensibility model and Chain of Command", "source_type": "microsoft_guidance", "module": "General", "approved": True,
     "text": "Dynamics 365 Finance and Operations is customized through extensions. Overlayering of Microsoft models is not supported, so standard objects are changed only through extension objects placed in a separate model. Tables, forms, data entities, enums and menus are extended with extension elements, for example a table extension that adds new fields. Business logic is extended with Chain of Command (CoC): a final extension class decorated with ExtensionOf wraps a public or protected method and must call next to run the standard logic. Event handlers (pre/post events and data events such as onValidatedWrite) are an alternative for reacting to standard events. Extensions keep the solution upgradable through Microsoft's continuous updates."},
    {"doc_id": "MS-SEC-01", "title": "Role-based security and segregation of duties", "source_type": "microsoft_guidance", "module": "General", "approved": True,
     "text": "Security in F&O is role based. Users are assigned security roles, roles contain duties, duties contain privileges, and privileges grant access levels to entry points such as menu items, and to tables and data entities. New menu items, forms or actions require new privileges, which are grouped into duties and added to roles through extensions rather than by changing standard roles. Segregation of duties (SoD) rules in System administration detect or prevent a user from holding conflicting duties, for example entering and approving vendor invoices. Workflow approvals should be assigned to roles or positions instead of named users so they keep working when people change."},
    {"doc_id": "MS-WF-01", "title": "Workflow and approvals", "source_type": "microsoft_guidance", "module": "Finance", "approved": True,
     "text": "F&O provides configurable approval workflows for documents such as vendor invoices, purchase requisitions, purchase orders and general journals. A workflow is configured per legal entity in the relevant module, for example Accounts payable > Setup > Accounts payable workflows. Approval steps can have conditions, such as the invoice total being greater than an amount, so that only documents above a threshold require approval. Assignments can target a user, a role, a position or a hierarchy. Each step supports time limits and escalation paths, for example escalate to another user after a number of days, and optional delegation. The workflow history records who submitted, approved or rejected a document and when, which supports audit requirements. When the vendor invoice workflow is active, invoices cannot be posted until the workflow is approved."},
    {"doc_id": "MS-AP-01", "title": "Vendor invoices and holds", "source_type": "microsoft_guidance", "module": "Finance", "approved": True,
     "text": "Vendor invoices are registered in the vendor invoice journal or the pending vendor invoices page, or created from purchase orders. Invoice matching validates invoices against purchase orders and product receipts (two-way or three-way matching) within configured price and quantity tolerances. A vendor can be placed on hold with the vendor's On hold field: Invoice blocks invoice entry and posting, Payment blocks payment generation while invoices can still be registered, and All blocks all transactions. Vendor bank accounts are maintained on the vendor record, and one of them can be selected as the vendor's default bank account. Changes to vendor master data can be controlled with the vendor approval (proposed changes) feature."},
    {"doc_id": "MS-GL-01", "title": "General ledger foundations", "source_type": "microsoft_guidance", "module": "Finance", "approved": True,
     "text": "The general ledger uses a chart of accounts shared across legal entities, main accounts and financial dimensions to build ledger accounts. Posting profiles map subledger transactions to main accounts. Ledger calendars and fiscal periods control which periods are open for posting. Number sequences provide document numbers per legal entity and should be configured rather than hard-coded."},
    {"doc_id": "MS-DATA-01", "title": "Data management and integration", "source_type": "microsoft_guidance", "module": "General", "approved": True,
     "text": "The data management framework uses data entities for import and export. Data projects can run once or as recurring integrations, and data packages can move configuration between environments. OData endpoints expose public data entities for lightweight real-time integration; large volumes should use the package API or recurring integrations. A custom data entity needs a staging table and is created with the data entity wizard in Visual Studio."},
    {"doc_id": "MS-BATCH-01", "title": "Batch processing with SysOperation", "source_type": "microsoft_guidance", "module": "General", "approved": True,
     "text": "Long-running or scheduled logic should run in the batch framework instead of on a form. The SysOperation framework separates a data contract (parameters), a controller (how the operation runs) and a service class (the business logic). Operations can be scheduled as recurring batch jobs, assigned to batch groups and monitored in System administration > Batch jobs. Batch logic should be idempotent so that a rerun does not create duplicate changes, and it should log the records it changes."},
    {"doc_id": "MS-PROC-01", "title": "Procurement and purchase order approval", "source_type": "microsoft_guidance", "module": "Supply Chain", "approved": True,
     "text": "Purchase requisitions let employees request goods and services; approved requisitions are released to purchase orders. Procurement categories classify purchases and drive policies. Change management on purchase orders activates the purchase order approval workflow, so a purchase order must be approved before it is confirmed and later changes require re-approval. Purchase agreements define committed quantities or amounts with vendors."},
    {"doc_id": "MS-INV-01", "title": "Inventory dimensions and batch control", "source_type": "microsoft_guidance", "module": "Supply Chain", "approved": True,
     "text": "Items are controlled by item model groups (costing method and inventory policies) and dimension groups. Storage dimensions include site, warehouse and location; tracking dimensions include batch and serial numbers. Batch-controlled items can store expiration dates, and inventory statuses or inventory blocking can prevent the use of specific on-hand quantities. Inventory closing settles issue and receipt transactions for the selected costing method."},

    # ---------------- Company standards ----------------
    {"doc_id": "STD-DEV-01", "title": "X++ development standard", "source_type": "company_standard", "module": "General", "approved": True,
     "text": "All custom objects use the prefix CUS and are placed in the model CUSExtensions. Customizations must use extensions and Chain of Command; overlayering and copying standard code are not allowed. Configurable values such as thresholds, days or amounts are stored in a parameter table exposed on a parameters form, never hard-coded. Every new menu item needs a privilege and a duty, delivered as extensions of existing roles. Business logic must have SysTest unit tests, and each change goes through pull-request code review in Azure DevOps."},
    {"doc_id": "STD-DOC-01", "title": "Design document standard", "source_type": "company_standard", "module": "General", "approved": True,
     "text": "A Functional Design Document (FDD) is required when a business process or configuration changes. It contains business context, as-is and to-be process, configuration and parameters, roles and security, exceptions, reports and data, assumptions and open points. A Technical Design Document (TDD) is required when code, integrations or reports are created. It contains solution overview, object list (extensions, classes, tables, security objects), logic, data model changes, error handling and logging, performance and batch considerations, and deployment steps. Mixed changes need both documents. Every section references the requirement IDs it covers."},
    {"doc_id": "STD-ADO-01", "title": "Azure DevOps work item standard", "source_type": "company_standard", "module": "General", "approved": True,
     "text": "Each approved requirement is delivered through at least one User Story. Story titles use the format '[Module] short goal', and the description uses 'As a <role>, I want <goal> so that <benefit>'. Acceptance criteria are written as Given/When/Then and reference requirement IDs. Implementation work is split into child Tasks: functional tasks are prefixed 'FUNC:' (configuration, FDD, training) and technical tasks are prefixed 'TECH:' (development, TDD, unit tests). Tasks carry an estimate in hours in Remaining Work. Stories are tagged with the module, work type and request ID. Assignees are proposed and then confirmed by a person; tools must not assign work automatically."},
    {"doc_id": "STD-TEST-01", "title": "Test case standard", "source_type": "company_standard", "module": "General", "approved": True,
     "text": "Each requirement is covered by at least one test case, and test cases reference requirement IDs. Test types: Functional (business scenario in the UI), Technical (unit or component behaviour, such as a SysTest), Integration (interaction with workflow, batch, data entities or external systems) and Regression (standard processes that must still work). Threshold rules need boundary tests at, just below and just above the limit. Security requirements need a negative test proving that an unauthorized user is blocked."},
    {"doc_id": "STD-COMM-01", "title": "Customer communication standard", "source_type": "company_standard", "module": "General", "approved": True,
     "text": "Clarification emails are short, polite and written in business language without technical jargon. Questions are numbered, one topic per question, with at most eight questions per email. Each email restates the request in one sentence, asks for a reply date, and never promises a delivery date or solution before the requirement is approved. Emails are saved as drafts and sent only after a person reviews them."},
    {"doc_id": "STD-CR-01", "title": "Change request standard", "source_type": "company_standard", "module": "General", "approved": True,
     "text": "A Change Request (CR) is required before any work that changes the agreed solution: new or changed functionality, customizations (X++ extensions), new or changed reports, integrations, data entities, workflows or fields, changes to the agreed scope, legal entities or business processes, and any configuration change outside the support contract. No CR is needed for support: defects in functionality that was delivered and accepted, errors caused by data or setup that are restored to the agreed design, how-to questions, access or security issues within the agreed roles, and standard Microsoft updates. When a request mixes both, the support part is handled as a ticket and the changed part as a CR. A CR uses the ERP Change/Additional Request Form: project name, change number, requested by, date of request, required phase, business priority, change name and technical complexity; the change description (business requirement and proposed solution) and the recommendation/best practice (customization required, affected areas, estimated effort) are written by the implementation partner; the business justification and the impact of not implementing the change are left for the customer's business to fill in; sign-off by the customer's functional lead, head of department and IT manager and by the partner's project manager. Work on a change starts only after the CR is signed."},

    # ---------------- Previously approved designs ----------------
    {"doc_id": "DES-2025-014", "title": "Approved design: purchase invoice approval by amount", "source_type": "approved_design", "module": "Finance", "approved": True,
     "text": "The vendor invoice workflow was activated per legal entity with the condition 'Invoice total (accounting currency) > threshold'. Invoices at or below the threshold were auto-approved with an automatic action so AP clerks could post them directly. The approval step was assigned to the Accounts payable manager role, with escalation to the finance director after 3 days. Segregation of duties was enforced with an SoD rule between the duties 'Maintain vendor invoices' and 'Approve vendor invoices', and the workflow configuration prevented the submitter from approving. The threshold was stored in a CUS parameter on the Accounts payable parameters form. Audit used the standard workflow history."},
    {"doc_id": "DES-2025-022", "title": "Approved design: capital purchase order approval", "source_type": "approved_design", "module": "Supply Chain", "approved": True,
     "text": "Purchase orders for procurement categories flagged as capital expenditure required approval by the plant controller. Change management was enabled for the affected legal entities, and the purchase order workflow used a condition on the procurement category. A CUS field on the procurement category table flagged capital categories. Regression tests covered non-capital purchase orders and purchase order changes after confirmation."},
    {"doc_id": "DES-2025-031", "title": "Approved design: automatic hold for vendors with incomplete master data", "source_type": "approved_design", "module": "Finance", "approved": True,
     "text": "A recurring SysOperation batch job (CUSVendMasterDataCheck) evaluated vendors in selected legal entities, set On hold = Payment when mandatory data was missing, and released the hold when the data was completed unless the hold had been set manually. A Chain of Command extension on the vendor table validated new vendors on insert and update. Each change was logged in a CUS log table with vendor, legal entity, old value, new value, reason and timestamp. The batch was idempotent and processed vendors per legal entity."},
    {"doc_id": "DES-DRAFT-007", "title": "Draft design (not approved): auto-post small invoices", "source_type": "approved_design", "module": "Finance", "approved": False,
     "text": "Draft idea to post vendor invoices below 500 USD automatically without any review. Rejected by the design authority and must not be reused."},

    # ---------------- Email templates ----------------
    {"doc_id": "TPL-CLARIFY-01", "title": "Clarification request template", "source_type": "email_template", "module": "General", "approved": True,
     "text": "Subject: Clarification needed - {request title}\n\nDear {customer name},\n\nThank you for your request regarding {one-sentence summary}. To make sure we deliver exactly what you need, could you please help us with the following questions:\n\n{numbered questions}\n\nCould you please reply by {reply date}? Once we have your answers, we will confirm the final scope before starting the work.\n\nKind regards,\n{consultant name}\nDynamics 365 F&O Delivery Team"},
    {"doc_id": "TPL-CONFIRM-01", "title": "Scope confirmation template", "source_type": "email_template", "module": "General", "approved": True,
     "text": "Subject: Scope confirmation - {request title}\n\nDear {customer name},\n\nThank you for your answers. Please find below the agreed requirements:\n\n{requirements}\n\nAssumptions:\n{assumptions}\n\nPlease confirm that this scope is correct so that we can start the design.\n\nKind regards,\n{consultant name}\nDynamics 365 F&O Delivery Team"},
]

print("Knowledge documents:", len(KNOWLEDGE_DOCS))

## Save the Documents as Files

A real team keeps its standards and designs as files. The documents are written to a `knowledge/` folder
and loaded back from there, so you can add your own `.md`, `.txt` or `.docx` files to the same folder.

In [ ]:
from pathlib import Path

KNOWLEDGE_DIR = Path("knowledge")
KNOWLEDGE_DIR.mkdir(exist_ok=True)

# Each document becomes a Markdown file with a small metadata header.
# You can drop your own .md / .txt / .docx files into this folder as well.
for doc in KNOWLEDGE_DOCS:
    header = "\n".join(f"{key}: {doc[key]}" for key in ("doc_id", "title", "source_type", "module", "approved"))
    (KNOWLEDGE_DIR / f"{doc['doc_id']}.md").write_text(f"---\n{header}\n---\n{doc['text']}\n", encoding="utf-8")

print(sorted(p.name for p in KNOWLEDGE_DIR.iterdir()))

## Load and Chunk

Longer documents are split with `RecursiveCharacterTextSplitter` (700 characters, 100 overlap) so that a
search returns the relevant paragraph instead of a whole document. Every chunk keeps the metadata of its
document.

In [ ]:
import docx
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter


def load_knowledge_folder(folder: Path) -> list[Document]:
    """Load .md/.txt/.docx files. Metadata comes from the header; files without one count as approved company standards."""
    documents = []
    for path in sorted(folder.iterdir()):
        if path.suffix.lower() in (".md", ".txt"):
            raw = path.read_text(encoding="utf-8")
        elif path.suffix.lower() == ".docx":
            raw = "\n".join(p.text for p in docx.Document(path).paragraphs if p.text.strip())
        else:
            continue

        metadata = {"doc_id": path.stem, "title": path.stem, "source_type": "company_standard",
                    "module": "General", "approved": True}
        if raw.startswith("---"):
            _, header, raw = raw.split("---", 2)
            for line in header.strip().splitlines():
                key, value = line.split(":", 1)
                metadata[key.strip()] = value.strip()
            metadata["approved"] = str(metadata["approved"]).lower() == "true"

        documents.append(Document(page_content=raw.strip(), metadata=metadata))
    return documents


splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=100)

source_documents = load_knowledge_folder(KNOWLEDGE_DIR)
chunks = splitter.split_documents(source_documents)
for doc_id in {c.metadata["doc_id"] for c in chunks}:
    for index, chunk in enumerate(c for c in chunks if c.metadata["doc_id"] == doc_id):
        chunk.metadata["chunk"] = index

print("Documents loaded:", len(source_documents))
print("Chunks created:", len(chunks))

## Create the Embeddings and the Vector Store

`OllamaEmbeddings` uses the local `nomic-embed-text` model. `InMemoryVectorStore` keeps the vectors in memory
and can be saved to a JSON file, which the MCP server loads later.

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model=EMBED_MODEL)

vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(chunks)

# Save the store so the knowledge MCP server can load it without re-embedding everything.
vector_store.dump("fo_knowledge_store.json")
print("Chunks embedded and saved to fo_knowledge_store.json")

## Search with Metadata Filters

`similarity_search` accepts a filter function. The search helper always removes unapproved documents and can
narrow the results to a module or a source type.

In [ ]:
def search_knowledge(query: str, module: str = "", source_type: str = "", k: int = 4):
    def keep(doc) -> bool:
        m = doc.metadata
        module_ok = module not in ("Finance", "Supply Chain") or m["module"] in (module, "General")
        return m["approved"] and module_ok and (not source_type or m["source_type"] == source_type)

    return vector_store.similarity_search(query, k=k, filter=keep)


def print_results(results) -> None:
    for doc in results:
        m = doc.metadata
        print(f"[{m['doc_id']}] ({m['source_type']}, {m['module']}) {doc.page_content[:110]}...")

### Test: filters

1. A Finance question limited to previously approved designs.
2. A question that matches the rejected draft almost word for word. The draft must **not** be returned.

In [ ]:
print("========== Approved designs about invoice approval ==========")
print_results(search_knowledge("approve vendor invoices above an amount", module="Finance", source_type="approved_design"))

print("\n========== Query that matches the rejected draft ==========")
results = search_knowledge("automatically post small vendor invoices without review", k=6)
print_results(results)
print("\nRejected draft returned:", any(d.metadata["doc_id"] == "DES-DRAFT-007" for d in results))

## The RAG Prompt

The model receives the question and the retrieved chunks, each labelled with its `doc_id`. It must answer
only from that context, cite the IDs it used and say so when the answer is not in the context.

In [ ]:
RAG_SYSTEM = """
You are a Dynamics 365 F&O knowledge assistant.
Answer using ONLY the context below. After each statement, cite the document ID in brackets, e.g. [MS-WF-01].
If the answer cannot be found in the context, say exactly:
"The answer is not available in the approved knowledge base."
Do not use outside knowledge and do not invent information.
"""


@traceable(name="RAG answer")
def ask_rag(question: str, module: str = "", k: int = 4) -> str:
    retrieved = search_knowledge(question, module=module, k=k)

    print("========== RETRIEVED CHUNKS ==========\n")
    print_results(retrieved)

    context = "\n\n".join(f"[{d.metadata['doc_id']}] {d.page_content}" for d in retrieved)
    answer = ask_text(RAG_SYSTEM, f"Context:\n{context}\n\nQuestion:\n{question}")

    print("\n========== ANSWER ==========\n")
    print(answer)
    return answer

## Test 1: Finance question answered by the knowledge base

The answer should combine Microsoft guidance (workflow conditions, workflow history) with the previously
approved design, and cite both.

In [ ]:
answer_1 = ask_rag(
    "How can we make vendor invoices above an amount require approval, and how can auditors see who approved them?",
    module="Finance",
)

## Test 2: Supply Chain question

With `module="Supply Chain"`, Finance-only documents are excluded while general standards remain available.

In [ ]:
answer_2 = ask_rag("How can we stop expired batches from being used?", module="Supply Chain")

## Test 3: Question not answered by the knowledge base

Nothing in the approved knowledge describes Power BI. The assistant must not invent an answer.

In [ ]:
answer_3 = ask_rag("How do we configure the embedded Power BI reports for the CFO dashboard?")

# Knowledge MCP Server

In the full system the agents do not import the vector store directly. They use an **MCP server** that
exposes two tools:

1. `search_fo_knowledge` - semantic search with module and source-type filters (approved documents only).
2. `get_document` - returns a full document, for example an email template.

The server loads the saved vector store, so it does not re-embed the documents. Log lines go to **stderr**,
because stdout is used by the MCP protocol.

In [ ]:
%%writefile knowledge_server.py
"""Knowledge MCP server: semantic search over approved F&O knowledge."""
import json
import os
import sys

from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import OllamaEmbeddings
from mcp.server.fastmcp import FastMCP

store = InMemoryVectorStore.load(
    "fo_knowledge_store.json", OllamaEmbeddings(model=os.getenv("EMBED_MODEL", "nomic-embed-text"))
)
mcp = FastMCP("fo_knowledge")


def log(message: str) -> None:
    print(f"[knowledge tool] {message}", file=sys.stderr, flush=True)  # stdout is reserved for MCP


@mcp.tool()
def search_fo_knowledge(query: str, module: str = "", source_type: str = "", k: int = 4) -> str:
    """Search approved Dynamics 365 F&O knowledge: Microsoft guidance, company standards,
    previously approved designs and email templates.
    module: '' (all), 'Finance' or 'Supply Chain'.
    source_type: '' (all), 'microsoft_guidance', 'company_standard', 'approved_design' or 'email_template'."""
    log(f"search_fo_knowledge(query={query!r}, module={module!r}, source_type={source_type!r})")

    def keep(doc) -> bool:
        m = doc.metadata
        module_ok = module not in ("Finance", "Supply Chain") or m["module"] in (module, "General")
        return bool(m["approved"]) and module_ok and (not source_type or m["source_type"] == source_type)

    results = store.similarity_search(query, k=k, filter=keep)
    return json.dumps([
        {"doc_id": d.metadata["doc_id"], "title": d.metadata["title"], "source_type": d.metadata["source_type"],
         "module": d.metadata["module"], "snippet": d.page_content}
        for d in results
    ])


@mcp.tool()
def get_document(doc_id: str) -> str:
    """Return the full text of one approved knowledge document (for example an email template) by its ID."""
    log(f"get_document({doc_id!r})")
    parts = sorted(
        (r for r in store.store.values() if r["metadata"]["doc_id"] == doc_id and r["metadata"]["approved"]),
        key=lambda r: r["metadata"].get("chunk", 0),
    )
    if not parts:
        raise ValueError(f"Unknown or unapproved document: {doc_id}")
    return json.dumps({"doc_id": doc_id, "title": parts[0]["metadata"]["title"],
                       "text": "\n".join(r["text"] for r in parts)})


if __name__ == "__main__":
    mcp.run(transport="stdio")

## MCP Client Helpers

These helpers start the server over `stdio`, discover its tools and call them. They run the asynchronous MCP
client in a separate event loop, so the same code works on Colab and in Jupyter on Windows.

In [ ]:
import asyncio
import contextlib
import contextvars
import json
import os
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

from langchain_mcp_adapters.tools import load_mcp_tools
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

MCP_LOG = Path("mcp_server.log")  # the servers' stderr ([tool called] lines)


def mcp_server(script: str) -> dict:
    return {"script": str(Path(script).resolve())}


@contextlib.asynccontextmanager
async def mcp_session(server: str):
    """Start a local FastMCP server over stdio and open an MCP session with it.

    The server's stderr goes to mcp_server.log: a notebook's own stderr cannot be handed to a subprocess.
    The whole environment is passed on (MCP only forwards a few safe variables by default), so the servers
    receive the Ollama and Azure DevOps settings.
    """
    params = StdioServerParameters(
        command=sys.executable, args=[MCP_SERVERS[server]["script"]], cwd=str(Path.cwd()), env=dict(os.environ)
    )
    with open(MCP_LOG, "a", encoding="utf-8") as errlog:
        async with stdio_client(params, errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                yield session


def run_async(coro):
    """Run a coroutine from a notebook cell and print the servers' tool log lines.

    It uses a fresh event loop in a worker thread, so it also works in Jupyter on Windows,
    where the notebook's own loop cannot start subprocesses.
    """
    def _run():
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        try:
            return loop.run_until_complete(coro)
        finally:
            loop.close()

    log_start = MCP_LOG.stat().st_size if MCP_LOG.exists() else 0
    context = contextvars.copy_context()  # keeps MCP tool calls inside the current LangSmith trace
    with ThreadPoolExecutor(max_workers=1) as pool:
        result = pool.submit(context.run, _run).result()

    if MCP_LOG.exists():
        with open(MCP_LOG, encoding="utf-8") as log:
            log.seek(log_start)
            for line in log.read().splitlines():
                if line.startswith("[") and "tool" in line.split("]")[0]:
                    print(line)
    return result


def _to_python(result):
    if isinstance(result, list):  # recent adapters return a list of content blocks
        result = "".join(b.get("text", "") if isinstance(b, dict) else str(b) for b in result)
    try:
        return json.loads(result)
    except (TypeError, json.JSONDecodeError):
        return result


def call_mcp_tools(server: str, calls: list) -> list:
    """Open one MCP session and run several (tool_name, arguments) calls in it."""
    async def _calls():
        async with mcp_session(server) as session:
            tools = {t.name: t for t in await load_mcp_tools(session)}
            results = []
            for name, args in calls:
                try:
                    results.append(_to_python(await tools[name].ainvoke(args)))
                except Exception as error:
                    print(f"[mcp error] {server}.{name}: {error}")
                    results.append({"error": str(error)})
            return results

    return run_async(_calls())


def call_mcp_tool(server: str, tool_name: str, **arguments):
    return call_mcp_tools(server, [(tool_name, arguments)])[0]


def list_mcp_tools(server: str) -> None:
    """tools/list: print the tools a server exposes, with their descriptions and inputs."""
    async def _list():
        async with mcp_session(server) as session:
            return (await session.list_tools()).tools

    print(f"========== TOOLS DISCOVERED ON '{server}' ==========\n")
    for tool in run_async(_list()):
        print(f"Name: {tool.name}")
        print(f"Description: {' '.join(tool.description.split())}")
        print(f"Inputs: {list(tool.inputSchema.get('properties', {}))}\n")
MCP_SERVERS = {"knowledge": mcp_server("knowledge_server.py")}

list_mcp_tools("knowledge")

### Calling the tools through MCP

The client does not know how the tools work. It only calls them by name with arguments that match the schemas
it discovered.

In [ ]:
hits = call_mcp_tool("knowledge", "search_fo_knowledge",
                     query="vendor on hold payment bank account", module="Finance", k=3)
for hit in hits:
    print(f"[{hit['doc_id']}] {hit['title']}")

template = call_mcp_tool("knowledge", "get_document", doc_id="TPL-CLARIFY-01")
print("\n" + template["text"])

## An Agent that Uses the Knowledge MCP Server

As in the MCP assignment, the tools are given to a LangGraph agent (`create_agent`). The agent decides when
to search, reads the observation and answers with citations. This is the ReAct loop with tools that arrive
through MCP.

In [ ]:
from langchain.agents import create_agent


async def ask_knowledge_agent(question: str):
    async with mcp_session("knowledge") as session:
        tools = await load_mcp_tools(session)  # discovered through MCP, not imported

        agent = create_agent(
            make_llm(),
            tools,
            system_prompt=(
                "You are a Dynamics 365 F&O knowledge assistant. Always call search_fo_knowledge before answering, "
                "answer only from the tool results and cite document IDs in brackets."
            ),
        )
        return await agent.ainvoke({"messages": [{"role": "user", "content": question}]})


result = run_async(ask_knowledge_agent(
    "What does our company standard say about how Azure DevOps tasks must be named?"
))
for message in result["messages"]:
    message.pretty_print()

## Observation

The knowledge base turns the organization's memory into something the agents can search. Metadata makes the
search controllable: results can be limited to a module or a source type, and the rejected draft
`DES-DRAFT-007` is never returned, even when the query matches it almost word for word.

For the Finance question, the answer was grounded in the Microsoft workflow guidance and the previously
approved design, and it cited them. For the Power BI question, nothing relevant was retrieved and the model
said that the answer is not available instead of inventing one.

The same search is exposed by `knowledge_server.py`. The client discovered the tools through MCP and the agent
used them without any tool-specific code. In the final notebook, the Gap and Retrieval Agent uses this server
to collect evidence, and the Communication Agent uses `get_document` to fetch the email template.